In [ ]:
import numpy as np
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score , confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import _perceptron


In [ ]:
import tensorflow as tf

In [ ]:
from  tensorflow.keras import models
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical

In [ ]:
df=pd.read_csv("animal_pixels.csv")
df.head()

In [ ]:
y=df["level"]
x=df.drop("level" , axis=1)


In [ ]:
animals=y.unique()
animal_level_en={}
i=0;
for animal in animals :
    animal_level_en[f"{animal}"]=i
    i+=1


In [ ]:
x_scaled=x/225.0
n=len(y.unique())
y=y.map(animal_level_en)


In [ ]:
y_encoded=to_categorical(y, num_classes=n)
y_encoded

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    x_scaled, y_encoded, test_size=0.25, random_state=42 , shuffle=True)


In [ ]:
X_test.shape

In [ ]:
y_test.shape

In [ ]:
X_test_img=X_test.values.reshape(-1,32,32,3)
X_train_img=X_train.values.reshape(-1,32,32,3)

In [ ]:
X_test_img.shape

#perceptron model

In [ ]:
perceptron_model=models.Sequential([
    layers.Flatten(input_shape=(32,32,3)),
    layers.Dense(90, activation="softmax")
])
    

In [ ]:
perceptron_model.compile(optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy'])

In [ ]:
history_per=perceptron_model.fit(X_train_img, y_train, validation_data=(X_test_img,y_test), epochs=20,
                                 batch_size=32, verbose=1)

In [ ]:
perceptron_model.evaluate(X_test_img,y_test,verbose=0)[1]

#CNN model

In [ ]:
cnn_model=models.Sequential([
    layers.Input(shape=(32,32,3)),
    layers.Conv2D(32, kernel_size=(3,3), activation="relu",padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2,2)),
    
    layers.Conv2D(64, kernel_size=(3,3), activation="relu",padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2,2)),
    
    layers.Conv2D(128, kernel_size=(3,3), activation="relu",padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2,2)),
    
    layers.Flatten(),
    layers.Dense(256,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(90,activation="softmax")
])

In [ ]:
cnn_model.compile(optimizer="adam" ,loss='categorical_crossentropy',metrics=['accuracy'])

In [ ]:
cnn_history=cnn_model.fit(
    X_train_img,y_train, 
    batch_size=32,          # Number of samples per gradient update
    epochs=80,              # Number of times to iterate over the entire dataset
    validation_data=(X_test_img, y_test), # Data to monitor validation loss/metrics
    shuffle=True, 
    verbose=1)

In [ ]:
cnn_model.evaluate(X_test_img,y_test)

In [ ]:
plt.figure(figsize=(20,20))
plt.plot(cnn_history.history["accuracy"])
plt.plot(cnn_history.history["val_accuracy"])
plt.legend(["Taining", "Validation"])
